# CNN training, weighted cross-entropy (final model)

Part of the code for *Danger Detection through Audio Captioning and Classification Fusion* (Electronics, MDPI). Paper section: 3.2, 3.2.1, 4.1.

Trains the raw-waveform CNN variants (LV-18, LV-24, LV-42, LV-81) and the feature-based CNN with weighted binary cross-entropy (class weights 4.18 : 1). The selected model is LV-18 with the validation AUC-ROC checkpoint. Screaming clips are augmented only in the training subset, after splitting.

**Environment.** Written for Google Colab (NVIDIA T4 GPU, TensorFlow 2.20.0, Keras 3.13.2); see `environment/`. Paths point to a Google Drive layout. Edit the path cell to your own copy of the datasets. Random seed: 42.

**Inputs.** Training manifest (2058 clips) and split files in data/splits/; ESC-50, UrbanSound8K and AudioSet screaming audio.

**Outputs.** Model checkpoints, internal test predictions (412 clips), UrbanSound8K predictions (8419 clips).


# New Danger Dataset CNN Training - Weighted Cross Entropy

Step-by-step training notebook for the new merged dataset using weighted binary cross entropy.

- ESC-50 danger classes: `dog`, `crying_baby`, `sneezing`, `coughing`, `glass_breaking`, `chainsaw`, `siren`, `car_horn`
- UrbanSound8K danger class: `gun_shot`
- Screaming samples: all available samples
- Non-danger classes: all remaining ESC-50 classes

Methodology:

1. Load the already-built `danger_non_danger_dataset.csv`.
2. Resolve Colab audio paths for ESC-50, UrbanSound8K, and screaming audio.
3. Split original samples first into train, validation, and test, stratified by the original class/category.
4. Keep validation and test untouched.
5. Augment only screaming samples in the training split until screaming has the same training count as the other danger classes.
6. Train with weighted binary cross entropy using `danger_weight = 4.18` and `no_danger_weight = 1.0`.
7. Use EarlyStopping and ReduceLROnPlateau instead of relying on a fixed final epoch.
8. Save the best model checkpoint for each run by validation loss and validation AUC.
9. Train the raw 1D waveform CNNs LV-18, LV-24, LV-42, LV-81 and the feature-extraction CNN.


## 1. Mount Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 2. Install / Import Dependencies

In [ ]:
from __future__ import annotations

import json
import math
import random
import shutil
import time
from pathlib import Path

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers, models
from tqdm import tqdm

print('TensorFlow:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))

## 3. Configure Paths and Training Settings

Change only this cell if your Google Drive folders are different.


In [ ]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

# Folder where you will place the CSV and cnn_training_common.py in Google Drive.
PROJECT_DIR = Path('/content/drive/MyDrive/Colab Notebooks/FINALS/NEW_DATASET/Weighted_CE_CNN')
ORIGINAL_PROJECT_DIR = Path('/content/drive/MyDrive/Colab Notebooks/FINALS/NEW_DATASET/CNNs')
DATASET_CSV = PROJECT_DIR / 'danger_non_danger_dataset.csv'
ORIGINAL_DATASET_CSV = ORIGINAL_PROJECT_DIR / 'danger_non_danger_dataset.csv'
HELPER_PY = PROJECT_DIR / 'cnn_training_common.py'

# Audio roots in Google Drive or /content after copying.
ESC50_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/ESC-50-master')
URBANSOUND_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/UrbanSound8k')
SCREAMING_WAV_DIR = Path('/content/drive/MyDrive/MyAudiosetData/screaming_final_5s_wav')

OUTPUT_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/FINALS/NEW_DATASET/Weighted_CE_CNN')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Reuse the exact split manifests from the previous focal-loss CNN training.
PREVIOUS_MANIFEST_DIR = Path('/content/drive/MyDrive/Colab Notebooks/FINALS/NEW_DATASET/CNNs/manifests')
REQUIRE_PREVIOUS_SPLITS = True

SR = 22050
DURATION_SECONDS = 5
TARGET_LENGTH = SR * DURATION_SECONDS
BATCH_SIZE = 32
EPOCH_BLOCK = 50

# Maximum epoch caps. EarlyStopping decides the actual stopping point.
WAVE_MAX_EPOCHS = {
    'LV-18': 100,
    'LV-24': 100,
    'LV-42': 100,
    'LV-81': 100,
}
FEATURE_MAX_EPOCHS = 100

EARLY_STOPPING_PATIENCE = 15
REDUCE_LR_PATIENCE = 7
REDUCE_LR_FACTOR = 0.5
MIN_LR = 1e-6
BEST_MODEL_MONITOR = 'val_loss'
TEST_SIZE = 0.20
VAL_SIZE_FROM_TRAINVAL = 0.20
NO_DANGER_WEIGHT = 1.0
DANGER_WEIGHT = 4.18

ARCHITECTURES = {
    'LV-18': (1, 8),
    'LV-24': (2, 4),
    'LV-42': (4, 2),
    'LV-81': (8, 1),
}

DANGER_CLASSES = {
    'dog', 'crying_baby', 'sneezing', 'coughing', 'glass_breaking',
    'chainsaw', 'siren', 'car_horn', 'gun_shot', 'screaming'
}

print('Project dir:', PROJECT_DIR)
print('Output dir:', OUTPUT_ROOT)


## 4. Optional: Import Existing Helper

The notebook does not depend on the old split helper for the new split, but it will reuse the existing export style if the helper file is available.


In [ ]:
import importlib.util

helper = None
if HELPER_PY.exists():
    spec = importlib.util.spec_from_file_location('cnn_training_common', HELPER_PY)
    helper = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(helper)
    print('Loaded helper:', HELPER_PY)
else:
    print('Helper not found. Continuing with notebook-local functions:', HELPER_PY)


def ensure_dir(path: Path | str) -> Path:
    out = Path(path)
    out.mkdir(parents=True, exist_ok=True)
    return out


def save_json(path: Path, payload: dict) -> None:
    with path.open('w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2)


## 5. Load Dataset and Resolve Colab Audio Paths

This cell maps the Windows paths in the CSV to Colab paths using `dataset_origin` and `filename`.


In [ ]:
def resolve_audio_path(row: pd.Series) -> str:
    origin = str(row['dataset_origin'])
    filename = str(row['filename'])

    if origin == 'ESC50':
        return str(ESC50_ROOT / 'audio' / filename)

    if origin == 'UrbanSound8K':
        fold = int(row['fold'])
        return str(URBANSOUND_ROOT / 'audio' / f'fold{fold}' / filename)

    if origin == 'Screaming':
        return str(SCREAMING_WAV_DIR / filename)

    raise ValueError(f'Unknown dataset_origin: {origin}')


dataset_csv_to_load = DATASET_CSV if DATASET_CSV.exists() else ORIGINAL_DATASET_CSV
if not dataset_csv_to_load.exists():
    raise FileNotFoundError(f'Dataset CSV not found in weighted folder or original folder: {DATASET_CSV} / {ORIGINAL_DATASET_CSV}')
print('Loading dataset CSV:', dataset_csv_to_load)
meta = pd.read_csv(dataset_csv_to_load)
meta['category'] = meta['category'].astype(str).str.strip()
meta['dataset_origin'] = meta['dataset_origin'].astype(str).str.strip()
meta['label_int'] = meta['danger_label'].astype(int)
meta['risk_label'] = np.where(meta['label_int'] == 1, 'danger risk', 'no danger risk')
meta['source_filename'] = meta['filename'].astype(str)
meta['audio_path'] = meta.apply(resolve_audio_path, axis=1)
meta['split_class'] = meta['category']
meta['is_augmented'] = False
meta['augmentation_type'] = 'original'
meta['source_index'] = np.arange(len(meta))

print('Rows:', len(meta))
print(meta.groupby(['dataset_origin', 'label_int']).size())
print('\nDanger class counts:')
print(meta[meta['label_int'] == 1]['category'].value_counts().sort_index())
meta.head()

## 6. Check Missing Audio Files

Run this before training. If anything is missing, fix the path config above or copy the audio folders into Drive.


In [ ]:
missing = meta[~meta['audio_path'].map(lambda p: Path(p).exists())].copy()
print('Total rows:', len(meta))
print('Missing audio files:', len(missing))

if len(missing):
    display(missing[['dataset_origin', 'category', 'filename', 'audio_path']].head(20))
    raise FileNotFoundError('Some audio files are missing. Fix paths before continuing.')
else:
    print('All audio files found.')

## 7. Split Original Samples First

This follows the previous training principle: split original samples before augmentation, and keep validation/test untouched.

Because this is now a mixed dataset with ESC-50 + UrbanSound8K + screaming, the split is made by the initial class/category with stratification. That is safer than the old helper's plain random split because it prevents tiny classes like `screaming` from disappearing from train/validation/test.


In [ ]:
def stratified_train_val_test_split(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    indices = np.arange(len(df))

    trainval_idx, test_idx = train_test_split(
        indices,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=df['split_class'],
    )

    trainval = df.iloc[trainval_idx].copy()
    train_idx_local, val_idx_local = train_test_split(
        np.arange(len(trainval)),
        test_size=VAL_SIZE_FROM_TRAINVAL,
        random_state=RANDOM_STATE,
        stratify=trainval['split_class'],
    )

    train = trainval.iloc[train_idx_local].copy().reset_index(drop=True)
    val = trainval.iloc[val_idx_local].copy().reset_index(drop=True)
    test = df.iloc[test_idx].copy().reset_index(drop=True)
    return train, val, test


def load_previous_split_manifests(manifest_dir: Path):
    required = {
        'train_original': manifest_dir / 'train_manifest_original_only.csv',
        'fit': manifest_dir / 'train_manifest_with_screaming_aug.csv',
        'val': manifest_dir / 'val_manifest.csv',
        'test': manifest_dir / 'test_manifest.csv',
    }
    missing = [str(path) for path in required.values() if not path.exists()]
    if missing:
        if REQUIRE_PREVIOUS_SPLITS:
            raise FileNotFoundError(
                'Previous split manifests are required so this run uses the exact same split. Missing:\\n'
                + '\\n'.join(missing)
            )
        print('Previous split manifests not found; recreating split with RANDOM_STATE:', RANDOM_STATE)
        train, val, test = stratified_train_val_test_split(meta)
        return train, None, val, test, 'recreated_from_seed'

    train = pd.read_csv(required['train_original'])
    fit = pd.read_csv(required['fit'])
    val = pd.read_csv(required['val'])
    test = pd.read_csv(required['test'])
    return train, fit, val, test, 'loaded_previous_focal_manifests'


def split_key_set(df: pd.DataFrame) -> set:
    key_cols = ['dataset_origin', 'category', 'source_filename', 'source_index']
    available = [col for col in key_cols if col in df.columns]
    return set(map(tuple, df[available].astype(str).to_numpy()))


train_manifest_original, fit_manifest_loaded, val_manifest, test_manifest, split_source = load_previous_split_manifests(PREVIOUS_MANIFEST_DIR)

train_keys = split_key_set(train_manifest_original)
val_keys = split_key_set(val_manifest)
test_keys = split_key_set(test_manifest)
if train_keys & val_keys or train_keys & test_keys or val_keys & test_keys:
    raise ValueError('Split leakage detected: train, validation, and test manifests overlap.')

print('Split source:', split_source)
print('Previous manifest dir:', PREVIOUS_MANIFEST_DIR)
print('Original train:', train_manifest_original.shape)
print('Validation:', val_manifest.shape)
print('Test:', test_manifest.shape)
if fit_manifest_loaded is not None:
    print('Loaded augmented training manifest:', fit_manifest_loaded.shape)
print('\nTrain category counts for danger classes before augmentation:')
print(train_manifest_original[train_manifest_original['label_int'] == 1]['category'].value_counts().sort_index())
print('\nValidation danger counts:')
print(val_manifest[val_manifest['label_int'] == 1]['category'].value_counts().sort_index())
print('\nTest danger counts:')
print(test_manifest[test_manifest['label_int'] == 1]['category'].value_counts().sort_index())


## 8. Augment Only Screaming Samples in Training

This creates manifest rows for augmented screaming clips only. It does not augment ESC-50 or UrbanSound8K.


In [ ]:
def add_noise(y: np.ndarray, noise_factor: float = 0.005) -> np.ndarray:
    noise = np.random.randn(len(y))
    return (y + noise_factor * noise).astype(np.float32)


def shift_audio(y: np.ndarray, shift_max: int = 1000) -> np.ndarray:
    shift = np.random.randint(-shift_max, shift_max)
    if shift > 0:
        y_shifted = np.pad(y, (shift, 0), mode='constant')[:len(y)]
    else:
        y_shifted = y[-shift:] if abs(shift) < len(y) else np.pad(y, (0, -shift), mode='constant')
    return y_shifted.astype(np.float32)


def pitch_shift_audio(y: np.ndarray, sr: int, n_steps: float = 2.0) -> np.ndarray:
    return librosa.effects.pitch_shift(y, sr=sr, n_steps=n_steps).astype(np.float32)


def time_stretch_audio(y: np.ndarray, rate: float = 1.1) -> np.ndarray:
    return librosa.effects.time_stretch(y, rate=rate).astype(np.float32)


SCREAMING_AUGMENTATION_SEQUENCE = [
    'noise', 'shift', 'pitch_up', 'pitch_down', 'stretch_fast', 'stretch_slow',
    'noise_shift', 'noise_pitch_up', 'noise_pitch_down', 'noise_stretch_fast'
]


def expand_training_with_screaming_augmentation(train_manifest: pd.DataFrame) -> pd.DataFrame:
    train = train_manifest.copy().reset_index(drop=True)
    danger_counts = train[train['label_int'] == 1]['category'].value_counts()
    non_screaming_danger_counts = danger_counts.drop(labels=['screaming'], errors='ignore')
    target_count = int(non_screaming_danger_counts.max())
    current_screaming = train[(train['label_int'] == 1) & (train['category'] == 'screaming')].copy()

    if current_screaming.empty:
        raise ValueError('No screaming samples landed in the training split. Adjust split settings.')

    needed = max(0, target_count - len(current_screaming))
    augmented_rows = []
    for i in range(needed):
        row = current_screaming.iloc[i % len(current_screaming)].copy()
        aug_name = SCREAMING_AUGMENTATION_SEQUENCE[i % len(SCREAMING_AUGMENTATION_SEQUENCE)]
        row['filename'] = f"{row['source_filename']}__aug_{i:03d}_{aug_name}"
        row['is_augmented'] = True
        row['augmentation_type'] = aug_name
        augmented_rows.append(row.to_dict())

    if augmented_rows:
        train = pd.concat([train, pd.DataFrame(augmented_rows)], ignore_index=True)

    print('Target screaming training count:', target_count)
    print('Original screaming in train:', len(current_screaming))
    print('Added screaming augmentations:', len(augmented_rows))
    return train.reset_index(drop=True)


if fit_manifest_loaded is not None:
    fit_manifest = fit_manifest_loaded.copy().reset_index(drop=True)
    print('Using previously saved augmented training manifest exactly:', fit_manifest.shape)
else:
    fit_manifest = expand_training_with_screaming_augmentation(train_manifest_original)

print('\nTraining danger counts after augmentation:')
print(fit_manifest[fit_manifest['label_int'] == 1]['category'].value_counts().sort_index())
print('\nTrain labels after augmentation:', np.bincount(fit_manifest['label_int'].astype(int)))

## 9. Save Split Manifests

In [ ]:
manifest_dir = ensure_dir(OUTPUT_ROOT / 'manifests')
meta.to_csv(manifest_dir / 'full_metadata_resolved.csv', index=False)
train_manifest_original.to_csv(manifest_dir / 'train_manifest_original_only.csv', index=False)
fit_manifest.to_csv(manifest_dir / 'train_manifest_with_screaming_aug.csv', index=False)
val_manifest.to_csv(manifest_dir / 'val_manifest.csv', index=False)
test_manifest.to_csv(manifest_dir / 'test_manifest.csv', index=False)
save_json(manifest_dir / 'split_source.json', {
    'split_source': split_source,
    'previous_manifest_dir': str(PREVIOUS_MANIFEST_DIR),
    'require_previous_splits': REQUIRE_PREVIOUS_SPLITS,
})

print('Saved manifests to:', manifest_dir)

## 9B. Save the Test Dataset

This saves the untouched original-only test split as a CSV, and can also copy the audio files into one folder if you want a portable test package.


In [ ]:
TEST_DATASET_DIR = ensure_dir(OUTPUT_ROOT / 'test_dataset')
TEST_AUDIO_DIR = ensure_dir(TEST_DATASET_DIR / 'audio')

# CSV version: this is usually enough because it includes labels, origin, caption, and resolved Colab audio paths.
test_dataset_csv = TEST_DATASET_DIR / 'test_dataset.csv'
test_manifest.to_csv(test_dataset_csv, index=False)

# Optional portable copy: set COPY_TEST_AUDIO = True if you want the test wav files copied next to the CSV.
COPY_TEST_AUDIO = False

if COPY_TEST_AUDIO:
    import shutil

    copied_rows = []
    for _, row in tqdm(test_manifest.iterrows(), total=len(test_manifest), desc='Copying test audio'):
        src = Path(row['audio_path'])
        safe_origin = str(row['dataset_origin']).replace(' ', '_')
        dst_name = f"{safe_origin}__{row['category']}__{src.name}"
        dst = TEST_AUDIO_DIR / dst_name
        shutil.copy2(src, dst)

        out_row = row.copy()
        out_row['copied_audio_path'] = str(dst)
        copied_rows.append(out_row)

    copied_test_csv = TEST_DATASET_DIR / 'test_dataset_with_copied_audio.csv'
    pd.DataFrame(copied_rows).to_csv(copied_test_csv, index=False)
    print('Saved copied-audio test CSV:', copied_test_csv)

print('Saved test dataset CSV:', test_dataset_csv)
print('Rows:', len(test_manifest))
print(test_manifest.groupby(['dataset_origin', 'category', 'label_int']).size().to_string())

## 10. Audio Loading and Preprocessing

UrbanSound8K clips shorter than 5 seconds are padded with silence here. The same fixed 5-second logic is used for every source.


In [ ]:
def fix_length_5s(y: np.ndarray, target_len: int = TARGET_LENGTH) -> np.ndarray:
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)), mode='constant')
    else:
        y = y[:target_len]
    return y.astype(np.float32)


def load_fixed_5s(filepath: str | Path, sr: int = SR) -> np.ndarray:
    y, _ = librosa.load(filepath, sr=sr, mono=True)
    return fix_length_5s(y)


def apply_screaming_augmentation(y: np.ndarray, sr: int, augmentation_type: str) -> np.ndarray:
    if augmentation_type == 'noise':
        return add_noise(y)
    if augmentation_type == 'shift':
        return shift_audio(y)
    if augmentation_type == 'pitch_up':
        return pitch_shift_audio(y, sr, n_steps=2.0)
    if augmentation_type == 'pitch_down':
        return pitch_shift_audio(y, sr, n_steps=-2.0)
    if augmentation_type == 'stretch_fast':
        return time_stretch_audio(y, rate=1.1)
    if augmentation_type == 'stretch_slow':
        return time_stretch_audio(y, rate=0.9)
    if augmentation_type == 'noise_shift':
        return shift_audio(add_noise(y))
    if augmentation_type == 'noise_pitch_up':
        return pitch_shift_audio(add_noise(y), sr, n_steps=2.0)
    if augmentation_type == 'noise_pitch_down':
        return pitch_shift_audio(add_noise(y), sr, n_steps=-2.0)
    if augmentation_type == 'noise_stretch_fast':
        return time_stretch_audio(add_noise(y), rate=1.1)
    raise ValueError(f'Unknown augmentation type: {augmentation_type}')


def load_manifest_signal(row: pd.Series) -> np.ndarray:
    y = load_fixed_5s(row['audio_path'], sr=SR)
    if bool(row['is_augmented']):
        y = apply_screaming_augmentation(y, SR, str(row['augmentation_type']))
        y = fix_length_5s(y)
    return y.astype(np.float32)

## 11. Weighted Cross Entropy and Metrics Helpers

In [ ]:
def weighted_binary_crossentropy(no_danger_weight: float = NO_DANGER_WEIGHT, danger_weight: float = DANGER_WEIGHT):
    no_danger_weight = tf.constant(no_danger_weight, dtype=tf.float32)
    danger_weight = tf.constant(danger_weight, dtype=tf.float32)

    def loss(y_true, y_pred):
        y_pred_f = tf.clip_by_value(tf.cast(y_pred, tf.float32), keras.backend.epsilon(), 1.0 - keras.backend.epsilon())
        y_true_f = tf.reshape(tf.cast(y_true, tf.float32), tf.shape(y_pred_f))
        bce = -(y_true_f * tf.math.log(y_pred_f) + (1.0 - y_true_f) * tf.math.log(1.0 - y_pred_f))
        weights = tf.where(tf.equal(y_true_f, 1.0), danger_weight, no_danger_weight)
        return tf.reduce_mean(weights * bce)

    return loss


WEIGHTED_BCE_LOSS = weighted_binary_crossentropy(
    no_danger_weight=NO_DANGER_WEIGHT,
    danger_weight=DANGER_WEIGHT,
)


def make_callbacks(run_dir: Path, monitor: str = BEST_MODEL_MONITOR):
    loss_model_path = run_dir / 'best_model_loss.keras'
    auc_model_path = run_dir / 'best_model_auc.keras'

    return [
        keras.callbacks.ModelCheckpoint(
            filepath=loss_model_path,
            monitor='val_loss',
            mode='min',
            save_best_only=True,
            save_weights_only=False,
            verbose=1,
        ),
        keras.callbacks.ModelCheckpoint(
            filepath=auc_model_path,
            monitor='val_auc_roc',
            mode='max',
            save_best_only=True,
            save_weights_only=False,
            verbose=1,
        ),
        keras.callbacks.EarlyStopping(
            monitor=monitor,
            mode='min' if 'loss' in monitor else 'max',
            patience=EARLY_STOPPING_PATIENCE,
            restore_best_weights=True,
            verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor=monitor,
            mode='min' if 'loss' in monitor else 'max',
            factor=REDUCE_LR_FACTOR,
            patience=REDUCE_LR_PATIENCE,
            min_lr=MIN_LR,
            verbose=1,
        ),
    ]


def load_saved_model(path: Path):
    return keras.models.load_model(
        path,
        custom_objects={'loss': WEIGHTED_BCE_LOSS},
    )


def evaluate_selected_models(
    run_dir: Path,
    x_test: np.ndarray,
    y_test: np.ndarray,
    x_val: np.ndarray,
    y_val: np.ndarray,
    test_manifest: pd.DataFrame,
    title_prefix: str,
):
    rows = []
    selected = []

    for selector, path in [
        ('val_loss', run_dir / 'best_model_loss.keras'),
        ('val_auc_roc', run_dir / 'best_model_auc.keras'),
    ]:
        if not path.exists():
            print('Missing selected model:', path)
            continue

        model = load_saved_model(path)
        val_eval = model.evaluate(x_val, y_val, verbose=0, return_dict=True)
        test_eval = model.evaluate(x_test, y_test, verbose=0, return_dict=True)

        selector_dir = ensure_dir(run_dir / f'selected_by_{selector}')
        probs = model.predict(x_test, verbose=0).reshape(-1)
        preds_df, cm_df, metrics_df = export_predictions(test_manifest, probs, selector_dir)
        plot_confusion(cm_df, selector_dir / 'test_confusion_matrix.png', f'{title_prefix} selected by {selector}')

        row = {
            'selector': selector,
            'model_path': str(path),
            **{f'val_{k}': float(v) for k, v in val_eval.items()},
            **{f'test_{k}': float(v) for k, v in test_eval.items()},
        }
        rows.append(row)
        selected.append((selector, model, metrics_df))
        tf.keras.backend.clear_session()

    summary_df = pd.DataFrame(rows)
    summary_df.to_csv(run_dir / 'selected_models_summary.csv', index=False)
    display(summary_df)
    return selected, summary_df

def export_predictions(test_manifest: pd.DataFrame, probs: np.ndarray, output_dir: Path, threshold: float = 0.5):
    output_dir = ensure_dir(output_dir)
    y_true = test_manifest['label_int'].to_numpy().astype(int)
    probs = probs.reshape(-1).astype(float)
    y_pred = (probs >= threshold).astype(int)

    preds = test_manifest.copy()
    preds['y_true'] = y_true
    preds['prob_danger_risk'] = probs
    preds['y_pred'] = y_pred
    preds['predicted_label'] = np.where(y_pred == 1, 'danger risk', 'no danger risk')
    preds['prediction_type'] = 'UNKNOWN'
    preds.loc[(preds['y_true'] == 1) & (preds['y_pred'] == 1), 'prediction_type'] = 'TP'
    preds.loc[(preds['y_true'] == 0) & (preds['y_pred'] == 0), 'prediction_type'] = 'TN'
    preds.loc[(preds['y_true'] == 0) & (preds['y_pred'] == 1), 'prediction_type'] = 'FP'
    preds.loc[(preds['y_true'] == 1) & (preds['y_pred'] == 0), 'prediction_type'] = 'FN'
    preds.to_csv(output_dir / 'test_predictions_detailed.csv', index=False)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    cm_df = pd.DataFrame(
        cm,
        index=['true_0_no_danger_risk', 'true_1_danger_risk'],
        columns=['pred_0_no_danger_risk', 'pred_1_danger_risk'],
    )
    cm_df.to_csv(output_dir / 'test_confusion_matrix.csv')

    report = classification_report(y_true, y_pred, labels=[0, 1], output_dict=True, zero_division=0)
    metrics_df = pd.DataFrame([
        ['accuracy', float(accuracy_score(y_true, y_pred))],
        ['TN', int(cm[0, 0])],
        ['FP', int(cm[0, 1])],
        ['FN', int(cm[1, 0])],
        ['TP', int(cm[1, 1])],
        ['class0_precision', float(report['0']['precision'])],
        ['class0_recall', float(report['0']['recall'])],
        ['class0_f1', float(report['0']['f1-score'])],
        ['class1_precision', float(report['1']['precision'])],
        ['class1_recall', float(report['1']['recall'])],
        ['class1_f1', float(report['1']['f1-score'])],
        ['auc_roc', float(roc_auc_score(y_true, probs)) if len(np.unique(y_true)) == 2 else float('nan')],
    ], columns=['metric', 'value'])
    metrics_df.to_csv(output_dir / 'test_metrics.csv', index=False)

    return preds, cm_df, metrics_df


def plot_confusion(cm_df: pd.DataFrame, out_path: Path, title: str) -> None:
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.show()


def plot_training_curves(history_df: pd.DataFrame, output_dir: Path, title_prefix: str) -> None:
    output_dir = ensure_dir(output_dir)

    plt.figure(figsize=(10, 5))
    plt.plot(history_df['epoch_global'], history_df['loss'], label='train_loss')
    plt.plot(history_df['epoch_global'], history_df['val_loss'], label='val_loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(f'{title_prefix} Loss')
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    loss_path = output_dir / 'training_validation_loss.png'
    plt.savefig(loss_path, dpi=200)
    plt.show()

    plt.figure(figsize=(10, 5))
    plt.plot(history_df['epoch_global'], history_df['accuracy'], label='train_accuracy')
    plt.plot(history_df['epoch_global'], history_df['val_accuracy'], label='val_accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.title(f'{title_prefix} Accuracy')
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    acc_path = output_dir / 'training_validation_accuracy.png'
    plt.savefig(acc_path, dpi=200)
    plt.show()

    print('Saved training plots:', loss_path, acc_path)


## 12. Build 1D Waveform Tensors

In [ ]:
def build_wave_tensor(manifest: pd.DataFrame) -> np.ndarray:
    x = np.zeros((len(manifest), TARGET_LENGTH, 1), dtype=np.float32)
    for i, row in enumerate(tqdm(manifest.to_dict('records'), desc='Loading waveforms')):
        signal = load_manifest_signal(pd.Series(row))
        x[i, :, 0] = signal
    return x


fit_x_wave = build_wave_tensor(fit_manifest)
val_x_wave = build_wave_tensor(val_manifest)
test_x_wave = build_wave_tensor(test_manifest)

fit_y = fit_manifest['label_int'].to_numpy().astype(np.float32)
val_y = val_manifest['label_int'].to_numpy().astype(np.float32)
test_y = test_manifest['label_int'].to_numpy().astype(np.float32)

print(fit_x_wave.shape, val_x_wave.shape, test_x_wave.shape)
print('Train labels:', np.bincount(fit_y.astype(int)))

## 13. Train 1D Waveform CNNs

In [ ]:
def create_lv_model(input_shape: tuple[int, int], stride: int, pool_size: int) -> tf.keras.Model:
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv1D(32, kernel_size=8, activation='relu', strides=stride),
        layers.MaxPooling1D(pool_size=pool_size),
        layers.Dropout(0.25),
        layers.Conv1D(64, kernel_size=8, activation='relu', strides=stride),
        layers.MaxPooling1D(pool_size=pool_size),
        layers.Dropout(0.25),
        layers.Conv1D(96, kernel_size=8, activation='relu', strides=stride),
        layers.MaxPooling1D(pool_size=pool_size),
        layers.Dropout(0.25),
        layers.GlobalAveragePooling1D(),
        layers.Dense(64, activation='relu'),
        layers.Dense(1, activation='sigmoid'),
    ])
    return model


def train_one_wave_architecture(name: str, stride: int, pool_size: int, max_epochs: int):
    run_dir = ensure_dir(OUTPUT_ROOT / '1DWAVE' / f'DangerDetective_1DWAVE_{name}_s{stride}_p{pool_size}')

    model = create_lv_model((TARGET_LENGTH, 1), stride, pool_size)
    model.compile(
        optimizer='adam',
        loss=WEIGHTED_BCE_LOSS,
        metrics=['accuracy', keras.metrics.Precision(name='precision'), keras.metrics.Recall(name='recall'), keras.metrics.AUC(name='auc_roc')],
    )

    start_time = time.time()
    history = model.fit(
        fit_x_wave, fit_y,
        epochs=max_epochs,
        batch_size=BATCH_SIZE,
        validation_data=(val_x_wave, val_y),
        callbacks=make_callbacks(run_dir),
        verbose=1,
    )

    history_df = pd.DataFrame(history.history)
    history_df.insert(0, 'epoch_global', np.arange(1, len(history_df) + 1))
    history_df.to_csv(run_dir / 'training_history_full.csv', index=False)
    plot_training_curves(history_df, run_dir, f'1D Wave {name}')

    selected_models, selected_summary_df = evaluate_selected_models(
        run_dir=run_dir,
        x_test=test_x_wave,
        y_test=test_y,
        x_val=val_x_wave,
        y_val=val_y,
        test_manifest=test_manifest,
        title_prefix=f'1D Wave {name}',
    )

    final_model_path = run_dir / f'DangerDetective_1DWAVE_{name}_s{stride}_p{pool_size}_early_stopped.keras'
    loss_model_path = run_dir / 'best_model_loss.keras'
    if loss_model_path.exists():
        shutil.copy2(loss_model_path, final_model_path)

    save_json(run_dir / 'run_config.json', {
        'model_family': '1DWAVE',
        'architecture': name,
        'stride': stride,
        'pool_size': pool_size,
        'max_epochs': max_epochs,
        'epochs_ran': int(len(history_df)),
        'batch_size': BATCH_SIZE,
        'random_state': RANDOM_STATE,
        'loss_function': 'weighted_binary_crossentropy',
        'no_danger_weight': NO_DANGER_WEIGHT,
        'danger_weight': DANGER_WEIGHT,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'reduce_lr_patience': REDUCE_LR_PATIENCE,
        'early_stopping_monitor': BEST_MODEL_MONITOR,
        'saved_best_models': ['best_model_loss.keras', 'best_model_auc.keras'],
        'training_seconds': time.time() - start_time,
    })

    print('Best loss model:', run_dir / 'best_model_loss.keras')
    print('Best AUC model:', run_dir / 'best_model_auc.keras')
    print('Exported loss-selected model saved to:', final_model_path)
    return selected_models, selected_summary_df


# Run all 1D-wave architectures. To test quickly, temporarily change this to a one-item dict.
wave_results = {}
for arch_name, (stride, pool_size) in ARCHITECTURES.items():
    print('\nRunning architecture:', arch_name)
    max_epochs = WAVE_MAX_EPOCHS[arch_name]
    print('Max epochs:', max_epochs)
    selected_models, selected_summary_df = train_one_wave_architecture(arch_name, stride, pool_size, max_epochs)
    wave_results[arch_name] = selected_summary_df
    tf.keras.backend.clear_session()

## 14. Build Feature Matrices

In [ ]:
def extract_mfcc(y: np.ndarray, sr: int) -> np.ndarray:
    return np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13), axis=1)


def extract_chroma(y: np.ndarray, sr: int) -> np.ndarray:
    return np.mean(librosa.feature.chroma_stft(y=y, sr=sr), axis=1)


def extract_spectral_contrast(y: np.ndarray, sr: int) -> np.ndarray:
    return np.mean(librosa.feature.spectral_contrast(y=y, sr=sr), axis=1)


def extract_feature_vector(y: np.ndarray, sr: int = SR) -> np.ndarray:
    return np.hstack([
        extract_mfcc(y, sr),
        extract_chroma(y, sr),
        extract_spectral_contrast(y, sr),
    ]).astype(np.float32)


def build_feature_matrix(manifest: pd.DataFrame) -> np.ndarray:
    x = np.zeros((len(manifest), 32), dtype=np.float32)
    for i, row in enumerate(tqdm(manifest.to_dict('records'), desc='Extracting features')):
        signal = load_manifest_signal(pd.Series(row))
        x[i, :] = extract_feature_vector(signal, SR)
    return x


fit_x_feat = build_feature_matrix(fit_manifest)
val_x_feat = build_feature_matrix(val_manifest)
test_x_feat = build_feature_matrix(test_manifest)

# Fit normalization on training only, then apply to val/test.
feature_mean = fit_x_feat.mean(axis=0, keepdims=True)
feature_std = fit_x_feat.std(axis=0, keepdims=True)
feature_std = np.where(feature_std == 0, 1.0, feature_std)
fit_x_feat = (fit_x_feat - feature_mean) / feature_std
val_x_feat = (val_x_feat - feature_mean) / feature_std
test_x_feat = (test_x_feat - feature_mean) / feature_std

print(fit_x_feat.shape, val_x_feat.shape, test_x_feat.shape)

In [ ]:
feature_run_dir = ensure_dir(OUTPUT_ROOT / 'FeatureExtract')


In [ ]:
def create_feature_model(input_dim: int) -> tf.keras.Model:
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Reshape((input_dim, 1)),

        layers.Conv1D(64, kernel_size=3, activation='relu'),
        layers.MaxPooling1D(pool_size=2),

        layers.Conv1D(128, kernel_size=3, activation='relu'),
        layers.MaxPooling1D(pool_size=2),

        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid'),
    ])
    return model


## 15. Train Feature-Extraction CNN

In [ ]:
feature_run_dir = ensure_dir(OUTPUT_ROOT / 'FeatureExtract')
feature_model = create_feature_model(fit_x_feat.shape[1])
feature_model.compile(
    optimizer='adam',
    loss=WEIGHTED_BCE_LOSS,
    metrics=['accuracy', keras.metrics.Precision(name='precision'), keras.metrics.Recall(name='recall'), keras.metrics.AUC(name='auc_roc')],
)

start_time = time.time()
history = feature_model.fit(
    fit_x_feat, fit_y,
    epochs=FEATURE_MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(val_x_feat, val_y),
    callbacks=make_callbacks(feature_run_dir),
    verbose=1,
)

history_df = pd.DataFrame(history.history)
history_df.insert(0, 'epoch_global', np.arange(1, len(history_df) + 1))
history_df.to_csv(feature_run_dir / 'training_history_full.csv', index=False)
plot_training_curves(history_df, feature_run_dir, 'Feature Extract')

selected_models, selected_summary_df = evaluate_selected_models(
    run_dir=feature_run_dir,
    x_test=test_x_feat,
    y_test=test_y,
    x_val=val_x_feat,
    y_val=val_y,
    test_manifest=test_manifest,
    title_prefix='Feature Extract',
)

final_feature_path = feature_run_dir / 'DangerDetective_FeatureExtract_NewDataset_early_stopped.keras'
loss_model_path = feature_run_dir / 'best_model_loss.keras'
if loss_model_path.exists():
    shutil.copy2(loss_model_path, final_feature_path)

np.save(feature_run_dir / 'feature_mean.npy', feature_mean)
np.save(feature_run_dir / 'feature_std.npy', feature_std)
save_json(feature_run_dir / 'run_config.json', {
    'model_family': 'FeatureExtract',
    'max_epochs': FEATURE_MAX_EPOCHS,
    'epochs_ran': int(len(history_df)),
    'batch_size': BATCH_SIZE,
    'random_state': RANDOM_STATE,
    'loss_function': 'weighted_binary_crossentropy',
    'no_danger_weight': NO_DANGER_WEIGHT,
    'danger_weight': DANGER_WEIGHT,
    'early_stopping_patience': EARLY_STOPPING_PATIENCE,
    'reduce_lr_patience': REDUCE_LR_PATIENCE,
    'early_stopping_monitor': BEST_MODEL_MONITOR,
    'saved_best_models': ['best_model_loss.keras', 'best_model_auc.keras'],
    'training_seconds': time.time() - start_time,
})

print('Best feature loss model:', feature_run_dir / 'best_model_loss.keras')
print('Best feature AUC model:', feature_run_dir / 'best_model_auc.keras')
print('Exported loss-selected feature model saved to:', final_feature_path)
display(selected_summary_df)

## 16. Quick Output Check

In [ ]:
print('Output root:', OUTPUT_ROOT)
for path in sorted(OUTPUT_ROOT.rglob('*')):
    if path.is_file() and path.suffix in {'.csv', '.json', '.keras', '.png', '.npy'}:
        print(path)

In [ ]:
# Collect final F1 scores for every trained model/selector.

from pathlib import Path
import pandas as pd

summary_rows = []

model_dirs = list((OUTPUT_ROOT / '1DWAVE').glob('DangerDetective_1DWAVE_*')) + [
    OUTPUT_ROOT / 'FeatureExtract'
]

for run_dir in model_dirs:
    summary_path = run_dir / 'selected_models_summary.csv'
    if not summary_path.exists():
        print('Missing summary:', summary_path)
        continue

    model_name = run_dir.name
    summary_df = pd.read_csv(summary_path)

    for _, row in summary_df.iterrows():
        selector = row['selector']
        metrics_path = run_dir / f'selected_by_{selector}' / 'test_metrics.csv'

        if not metrics_path.exists():
            print('Missing metrics:', metrics_path)
            continue

        metrics_df = pd.read_csv(metrics_path)
        metrics = dict(zip(metrics_df['metric'], metrics_df['value']))

        summary_rows.append({
            'model': model_name,
            'selector': selector,
            'test_accuracy': row.get('test_accuracy'),
            'test_auc_roc': row.get('test_auc_roc'),
            'test_precision': row.get('test_precision'),
            'test_recall': row.get('test_recall'),
            'test_f1_class0_no_danger': metrics.get('class0_f1'),
            'test_f1_class1_danger': metrics.get('class1_f1'),
            'TN': metrics.get('TN'),
            'FP': metrics.get('FP'),
            'FN': metrics.get('FN'),
            'TP': metrics.get('TP'),
        })

f1_summary_df = pd.DataFrame(summary_rows)
f1_summary_df = f1_summary_df.sort_values(
    by=['test_f1_class1_danger', 'test_auc_roc'],
    ascending=False
).reset_index(drop=True)

f1_summary_path = OUTPUT_ROOT / 'weighted_ce_all_models_f1_summary.csv'
f1_summary_df.to_csv(f1_summary_path, index=False)

display(f1_summary_df)
print('Saved F1 summary to:', f1_summary_path)